# 🎛️ Step 5 · Similarity-Search App (Gradio)

**Inference UI.** Paste an image URL → embed it → return the most similar faces from the online index.

In [ ]:
import gradio as gr
from PIL import Image
import requests
from io import BytesIO
import hopsworks
import torch
from transformers import CLIPProcessor, CLIPModel
import functools


def get_image_embedding(image: Image.Image, processor: CLIPProcessor, model: CLIPModel, device: str):
    try:
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.get_image_features(**inputs)
        # transformers >=5 returns a model output object; <5 returns a tensor
        image_features = getattr(out, "pooler_output", out)
        # L2-normalize
        image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
        return image_features.squeeze().cpu().tolist()
    except Exception as e:
        print(f"Error computing embedding: {e}")
        raise


# Hopsworks setup
proj = hopsworks.login()

fs = proj.get_feature_store()
fg = fs.get_feature_group("image_embeddings", version=1)
mr = proj.get_model_registry()
model_mr = mr.get_model("openaiclip_vit_base_patch32", version=1)
dir = model_mr.download()

# Load CLIP
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained(dir).to(device).eval()
processor = CLIPProcessor.from_pretrained(dir)


def load_image_from_url(url, model, processor, device, fg):
    try:
        # load query image
        response = requests.get(url)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content))

        # get embedding and neighbors
        embedding = get_image_embedding(image, processor, model, device)
        results = fg.find_neighbors(embedding, k=3)

        returned_files = []
        returned_images = []
        for result in results:
            path = result[1][0]  # adapt to actual structure
            returned_files.append(path)
            try:
                img = Image.open(path)
                returned_images.append(img)
            except Exception as e:
                print(f"Error opening {path}: {e}")
                returned_images.append(None)

        return (
            image,  # main image
            f"Format: {image.format}, Size: {image.size}, Mode: {image.mode}",
            returned_images,
            "\n".join(returned_files)
        )

    except Exception as e:
        return None, f"Error: {str(e)}", [], ""


# wrap function
load_image = functools.partial(load_image_from_url, model=model, processor=processor, device=device, fg=fg)


# Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## Load Image from URL and Find Similar Images")

    with gr.Row():
        url_input = gr.Textbox(label="Enter image URL")
        output_image = gr.Image(type="pil", label="Input Image")
        output_info = gr.Textbox(label="Image details")

    with gr.Row():
        similar_images = gr.Gallery(label="Similar Images", columns=3, rows=1)

    with gr.Row():
        similar_files = gr.Textbox(label="Similar Image Filenames")

    url_input.change(
        load_image,
        inputs=[url_input],
        outputs=[output_image, output_info, similar_images, similar_files]
    )

demo.launch(share=True)
